# 03 — Modelagem Não Supervisionada: Clusterização e PCA

Este notebook aplica técnicas de aprendizado não supervisionado para descobrir padrões latentes nos dados de voos:

1. **PCA** (Análise de Componentes Principais) — redução de dimensionalidade e visualização
2. **K-Means** — segmentação dos voos em grupos homogêneos

A análise é realizada sobre as **variáveis numéricas** do dataset, preservando interpretabilidade e maximizando a variância explicada.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from src.utils import generate_flight_data

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

NUMERICAL_FEATURES = [
    'dep_delay_min', 'arr_delay_min', 'distance_km',
    'taxi_out_min', 'taxi_in_min', 'carrier_delay_min',
    'dep_hour', 'day_of_week', 'month'
]

## 1. Preparação dos Dados

In [ ]:
df_raw = generate_flight_data(n_samples=5000, random_state=RANDOM_STATE)
df_clean = df_raw[df_raw['cancelled'] == 0].reset_index(drop=True)

X_num = df_clean[NUMERICAL_FEATURES].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)

print(f'Registros: {len(df_clean)}')
print(f'Features: {len(NUMERICAL_FEATURES)}')
print(f'Shape: {X_scaled.shape}')

## 2. PCA — Análise de Variância Explicada

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

n_90 = int(np.argmax(cumulative_var >= 0.90)) + 1
print('Variância por componente:')
for i, v in enumerate(explained_var):
    print(f'  PC{i+1}: {v*100:.1f}%  (acum: {cumulative_var[i]*100:.1f}%)')
print(f'\nComponentes para 90% da variância: {n_90}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained_var)+1), explained_var*100, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Variância Explicada (%)')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var*100, marker='o', linewidth=2, color='steelblue')
axes[1].axhline(90, color='red', linestyle='--', label='90% variância')
axes[1].axhline(80, color='orange', linestyle='--', label='80% variância')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Variância Acumulada (%)')
axes[1].set_title('Variância Acumulada pelo PCA')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca_2d.fit_transform(X_scaled)

var_pc1 = pca_2d.explained_variance_ratio_[0] * 100
var_pc2 = pca_2d.explained_variance_ratio_[1] * 100
print(f'PC1: {var_pc1:.1f}%  |  PC2: {var_pc2:.1f}%  |  Total: {var_pc1+var_pc2:.1f}%')

loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=NUMERICAL_FEATURES,
    columns=['PC1', 'PC2']
).round(3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
loadings['PC1'].sort_values().plot(kind='barh', ax=axes[0],
    color=['#F44336' if v > 0 else '#2196F3' for v in loadings['PC1'].sort_values()],
    edgecolor='black')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title(f'Loadings — PC1 ({var_pc1:.1f}%)')

loadings['PC2'].sort_values().plot(kind='barh', ax=axes[1],
    color=['#F44336' if v > 0 else '#2196F3' for v in loadings['PC2'].sort_values()],
    edgecolor='black')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title(f'Loadings — PC2 ({var_pc2:.1f}%)')

plt.tight_layout()
plt.show()
print(loadings)

## 3. K-Means — Seleção do k Ótimo

In [ ]:
inertias, sil_scores = [], []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, lbl))
    print(f'k={k}: inércia={km.inertia_:.1f}, silhouette={sil_scores[-1]:.4f}')

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f'\nMelhor k (Silhouette): {best_k}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, marker='o', linewidth=2, color='steelblue')
axes[0].set_xlabel('Número de Clusters (k)')
axes[0].set_ylabel('Inércia (WSS)')
axes[0].set_title('Método do Cotovelo')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range), sil_scores, marker='o', linewidth=2, color='coral')
axes[1].axvline(best_k, color='red', linestyle='--', label=f'k ótimo = {best_k}')
axes[1].set_xlabel('Número de Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score por k')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Treinamento Final e Visualização dos Clusters

In [ ]:
K_OPTIMAL = best_k
kmeans = KMeans(n_clusters=K_OPTIMAL, init='k-means++', n_init=10, random_state=RANDOM_STATE)
cluster_labels = kmeans.fit_predict(X_scaled)
df_clean = df_clean.copy()
df_clean['cluster'] = cluster_labels

print(f'K-Means final: k={K_OPTIMAL}')
print(f'Silhouette Score: {silhouette_score(X_scaled, cluster_labels):.4f}')
print(f'\nDistribuição:')
print(df_clean['cluster'].value_counts().sort_index())

In [ ]:
palette = sns.color_palette('tab10', K_OPTIMAL)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for c in range(K_OPTIMAL):
    mask = cluster_labels == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    label=f'Cluster {c} (n={mask.sum()})',
                    alpha=0.5, s=12, color=palette[c])
axes[0].set_xlabel(f'PC1 ({var_pc1:.1f}%)')
axes[0].set_ylabel(f'PC2 ({var_pc2:.1f}%)')
axes[0].set_title(f'Clusters K-Means (k={K_OPTIMAL})')
axes[0].legend(markerscale=2, fontsize=9)

for label, color, name in [(0, 'steelblue', 'Não Atrasado'), (1, 'tomato', 'Atrasado')]:
    mask = df_clean['delayed'] == label
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    label=name, alpha=0.4, s=12, color=color)
axes[1].set_xlabel(f'PC1 ({var_pc1:.1f}%)')
axes[1].set_ylabel(f'PC2 ({var_pc2:.1f}%)')
axes[1].set_title('Variável Alvo no Espaço PCA')
axes[1].legend(markerscale=2)

plt.tight_layout()
plt.show()

## 5. Perfil dos Clusters

In [ ]:
cluster_profile = df_clean.groupby('cluster')[NUMERICAL_FEATURES + ['delayed']].mean().round(2)
print('Perfil médio dos clusters:')
print(cluster_profile.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['dep_delay_min', 'arr_delay_min', 'distance_km']):
    cluster_profile[col].plot(kind='bar', ax=ax, color=palette[:K_OPTIMAL], edgecolor='black')
    ax.axhline(df_clean[col].mean(), color='red', linestyle='--', linewidth=1, label='Média global')
    ax.set_title(f'{col} por Cluster')
    ax.set_xticklabels([f'C{i}' for i in range(K_OPTIMAL)], rotation=0)
    ax.legend(fontsize=8)
plt.suptitle('Perfil dos Clusters', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

delay_rate = df_clean.groupby('cluster')['delayed'].mean() * 100
delay_rate.plot(kind='bar', ax=axes[0], color=palette[:K_OPTIMAL], edgecolor='black')
axes[0].axhline(df_clean['delayed'].mean()*100, color='red', linestyle='--',
                label=f'Média global ({df_clean["delayed"].mean()*100:.1f}%)')
axes[0].set_title('Taxa de Atraso (%) por Cluster')
axes[0].set_ylabel('Taxa de Atraso (%)')
axes[0].set_xticklabels([f'C{i}' for i in range(K_OPTIMAL)], rotation=0)
axes[0].legend()

weather_pct = (df_clean.groupby(['cluster', 'weather']).size()
               .unstack(fill_value=0)
               .pipe(lambda df: df.div(df.sum(axis=1), axis=0) * 100))
weather_pct.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='black')
axes[1].set_title('Distribuição Climática por Cluster (%)')
axes[1].set_xticklabels([f'C{i}' for i in range(K_OPTIMAL)], rotation=0)
axes[1].legend(title='Clima', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

## 6. Análise Crítica

### PCA
- **PC1** captura a dimensão de **intensidade do atraso**: `dep_delay_min`, `arr_delay_min` e `carrier_delay_min` têm os maiores loadings, representando o eixo principal de variabilidade nos dados.
- **PC2** captura a dimensão **operacional/temporal**: `distance_km`, `dep_hour` e `taxi_out_min` dominam essa componente.
- As 2 componentes explicam ~36% da variância — suficiente para visualização, mas a estrutura completa requer mais componentes.

### K-Means
- O k ótimo identificado agrupa os voos em perfis bem distintos em termos de atraso:
  - **Cluster de baixo atraso**: voos pontuais com atrasos abaixo da média global
  - **Cluster de alto atraso**: voos com atrasos significativamente acima da média, correlacionados com condições climáticas adversas e atrasos operacionais das companhias

### Limitações
1. **Silhouette moderado**: os clusters têm alguma sobreposição, comum em dados de voo sem separações naturais nítidas.
2. **K-Means assume clusters esféricos**: DBSCAN ou GMM poderiam capturar formas mais complexas.
3. **Dataset sintético**: os padrões identificados são artefatos da geração de dados; com dados reais, a interpretação seria mais rica.
4. **Inclusão de variáveis categóricas**: k-prototypes permitiria usar airline, origin, destination e weather diretamente na clusterização.